In [ ]:
# pip install pandas nltk wordcloud

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from nltk.stem import SnowballStemmer
import nltk
from nltk.corpus import reuters
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import re
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Text Analysis Problems

# 4.8        (Text Analysis)  How does TF-IDF enhance the relevance of a search result?

### TF-IDF (Term Frequency–Inverse Document Frequency / Частота термина – обратная частота документов)

TF-IDF ранжирует документы по близости к запросу: он представляет запрос и документы как вектор весов терминов и по ним измеряет близость к тому или иному документу

- ```TF``` - частота термина, сколько раз встречается слово в конкретном документе. Чем чаще запрос встречается в документе, тем выше вероятность связи темы документа с запросом
$$ TF(t, \ d) \ = \ (число \ вхождений \ t \ в \ d) \  / \ (общее \ число \ слов \ в \ d) $$
- ```IDF``` - обратная частота документа, показывает насколько редко слово встречается во всей коллекции документов
$$ IDF(t) = log(N / D(t)),$$ где N - общее число документов\
D(t) - число документов, содержащих t

Чем реже термин, тем больше он имеет вес: например, "на", "и" встречаются часто и потому их ```IDF``` будет стремиться к нулю, а редкие термины как "триангуляция" будут встречаться реже и иметь большее значение ```IDF```
*Опечатки также могут получать высокий IDF, в следствие чего результаты приходится фильтровать*

Итоговый вес термина высок, когда он часто встречается в документе и редок в коллекции документов
$$ TF-IDF \ = \ TF \ \cdot \ IDF $$

TF-IDF помогает оценить релевантность через частоту термина в документе относительно частоты этого термина в коллекции

## 4.9        (Text Analysis)  Why should we reduce the dimensions in text analysis? How to achieve that?

Почему мы должны уменьшать размерность:
- Разреженность: в каждом документе встречается лишь часть словаря, в связи с чем много элементов вектора равны нулю
- Переобучение: чем больше признаков при ограниченном числе документов, тем выше шансы словить переобучение
- Шум: если не уменьшать рaзмерность, оставим опечатки или слишком редкие термины, которые несут малую информационную значимость
- Оптимизация: работа с многомерными структурами является ресурсозатратной



Как это делается:
- Фильтрация по частоте (Удалить все слова, что встречаются реже/чаще порогового значения)
- Стемминг (Нахождение основы слова путем удаления суффиксов, приставок) и Лемматизация (Приведение словоформы к ее базовой версии: "кошками" -> "кошк"/"кошка")

## 4.10      (Text Analysis)  Yoyodyne Bank is an international bank that has customers from around the world. The bank plans to research the customer opinions using text analysis on social networks, such as Twitter, Facebook, and Google+.  By conducting the research Yoyodyne bank hopes to improve its services and bring in more customers.

### a.      How should Yoyodyne Bank do such a text analysis?


1. Определить цель текстового анализа
2. Сформулировать метрики, с помощью которых будет оцениваться полученный результат
3. Сформировать список сервисов, из которых можно получить нужные данные
4. Собрать данные и предобработать их
5. Разметить данные
6. Выбрать модель
7. Оценить качество результата метриками
8. Перевести результаты исследования в отчет


### b.      What are the advantages? What are the challenges?

advantages:
- Большой объем данных с дополнительной информацией о человеке (Социальный граф, профиль)
- Возможность оценить реакцию и с точки зрения временного промежутка между событием и постом
- Можно не только проанализировать посты моделью и определить мнение клиента, но и в дальнейшем их классифицировать по дополнительным признакам
- Нахождение потенциальных клиентов, чьи боли решаются нашим сервисом
- Возможность незатратно перевести модель из "сборки мнения про Yoyodyne bank" в "сборка мнения про банк-конкурент"
- Невысокие затраты при долгосрочном использовании

challenges:
- Доступ к данным
- "Качество мнения" - не каждое мнение в социальной сети является необходимым для отработки
- Определение ЦА среди пользователей сети
- Определение стилистической окраски текста: ирония, эмодзи, серьезность
- Разметка данных - никто не гарантирует, что нанятые для разметки специалисты справятся со своей работой на должном уровне
- Обработка высококонтекстных постов - поскольку мы глобальный банк, нам надо быть готовым не только к низкоконтекстным странам по типу Германии, где обрабатывать текста проще, но и к высококонтекстным странам по типу Японии

Разница между высококотекстной и низкоконтекстной страной на примере рекламы
| Низкоконтекстная Германия | Высококонтекстная Япония |
| :-: | :-: |
| ![](resources/9-lab/Germany_ad.jpg) | ![](resources/9-lab/Japan_ad.jpg)  |
------

## 4.11    (Text Analysis) Below is a review on Yoyodyne Bank captured from social media:
```
Sneaky Fees!

Do you carefully read every word on your statement and every notice your bank every sent you? If you've missed one, Yoyodyne Bank is NOT the bank for you! Close all your accounts especially if you're going overseas!!
```
### a.       If case-insensitive, what may be the “bag of words” form of this review when stop words are removed? You may use the Snowball English stop word list:

Snowball - проект Мартина Портера, в нем есть stop.txt - набор служебных слов, которые обычно удаляют при препроцессинге

In [ ]:
review = "Sneaky Fees! Do you carefully read every word on your statement and every notice your bank every sent you? If you've missed one, Yoyodyne Bank is NOT the bank for you! Close all your accounts especially if you're going overseas!!"

stop_words = set()
for line in open("resources/9-lab/stop.txt"):
    word = line.split("|")[0].strip().lower()
    if word: stop_words.add(word)

stop_words.add("one")
stop_words.add("every") # добавлены, тк являются одними 
# из самых распространенных слов (Cм конец файла stop.txt)

review = review.lower()
for i in "?!,":
    review = review.replace(i, "")

bag = Counter(review.split())
for i in stop_words:
    bag.pop(i, None)

bag

### b.      Based on the Step 1, what could be the form when stemming is enabled as well?
### You may refer to the Snowball stemming demo (http://snowballstem.org/demo.html) and the Porter stemming algorithm (http://tartarus.org/~martin/PorterStemmer).

SnowballStemmer.stem() - алгоритм по нормализации слова:
1. Делит на R1 - часть слова после первой гласной, за которой следует согласная и R2 - то же правило, но примененнео внутри R1
2. Удаляет притяжательные окончания ```'s```, ```'```, ```s'```
3. Множественное число (Удаление s на окончании) ```accounts -> account```
4. Удаление ```ed```, ```ing```
5. замена ```y``` на ```i```, если перед ним согласная и это не первая буква
6. Замена/удаление производных суффиксов, если те попадают в R1, R2 ```careflly -> careful```
7. Удаление финальной ```e```, елси она в ```R2``` или в ```R1``` и перед ней не короткий слог ```notice -> notic```
- Также есть и исключения, для которых правила не применяются, например: ```news```, ```skis```, ```sky```, ```only```


In [ ]:
stemmer = SnowballStemmer("english")

stemmed = Counter()
for i, j in bag.items():
    stemmed[stemmer.stem(i)] += j

stemmed

## 4.12    (Text Analysis)  In this problem, you are given a news article. Use frequencies and TF-IDF in R to identify the most important words of the article. Visualize them in word clouds.

### The first two lines load the tm and wordcloud libraries for R if they are installed. The third line loads the data set crude which holds 20 news articles with additional meta information from the Reuters-21578 XML data set. All documents belong to the topic crude dealing with crude oil.

### a.       The 2nd document in the crude corpus represents the following article; read through the article and write down 10 words that are representative of it.



```
OPEC may be forced to meet before a scheduled June session to readdress its production cutting agreement if the organization wants to halt the current slide in oil prices, oil industry analysts said.

    "The movement to higher oil prices was never to be as easy as OPEC thought. They may need an emergency meeting to sort out the problems," said Daniel Yergin, director of Cambridge Energy Research Associates, CERA.

    Analysts and oil industry sources said the problem OPEC faces is excess oil supply in world oil markets.

    "OPEC's problem is not a price problem but a production issue and must be addressed in that way," said Paul Mlotok, oil analyst with Salomon Brothers Inc.

    He said the market's earlier optimism about OPEC and its ability to keep production under control have given way to a pessimistic outlook that the organization must address soon if it wishes to regain the initiative in oil prices.

    But some other analysts were uncertain that even an emergency meeting would address the problem of OPEC production above the 15.8 mln bpd quota set last December.

    "OPEC has to learn that in a buyers market you cannot have deemed quotas, fixed prices and set differentials," said the regional manager for one of the major oil companies who spoke on condition that he not be named. "The market is now trying to teach them that lesson again," he added.

    David T. Mizrahi, editor of Mideast reports, expects OPEC to meet before June, although not immediately. However, he is not optimistic that OPEC can address its principal problems.

    "They will not meet now as they try to take advantage of the winter demand to sell their oil, but in late March and April when demand slackens," Mizrahi said.

    But Mizrahi said that OPEC is unlikely to do anything more than reiterate its agreement to keep output at 15.8 mln bpd."

    Analysts said that the next two months will be critical for OPEC's ability to hold together prices and output.

    "OPEC must hold to its pact for the next six to eight weeks since buyers will come back into the market then," said Dillard Spriggs of Petroleum Analysis Ltd in New York.

    But Bijan Moussavar-Rahmani of Harvard University's Energy and Environment Policy Center said that the demand for OPEC oil has been rising through the first quarter and this may have prompted excesses in its production.

    "Demand for their (OPEC) oil is clearly above 15.8 mln bpd and is probably closer to 17 mln bpd or higher now so what we are seeing characterized as cheating is OPEC meeting this demand through current production," he told Reuters in a telephone interview.

 Reuter
 ```

In [ ]:
# раскоментить, если нужно скачать

# nltk.download("reuters")

In [ ]:
article = open("resources/9-lab/2-nd_article.txt").read()
ids = reuters.fileids("crude")[:19]

docs = [reuters.raw(ids[0]), article] + [reuters.raw(i) for i in ids[1:]]

processed_docs = []
for doc in docs:
    doc = doc.lower()
    for i in "?!,.()\\'\"":
        doc = doc.replace(i, "")

    for i in "\n\t":
        doc = doc.replace(i, " ")

    processed_docs.append(doc)

    

processed_docs

In [ ]:
stop_words = {w.replace("'", "") for w in stop_words}
    
vectorizer = TfidfVectorizer(stop_words=list(stop_words), token_pattern=r"\b[a-zA-Z]{3,}\b")
vectorizer.fit(processed_docs)

In [ ]:
X = vectorizer.transform(processed_docs)
terms = vectorizer.get_feature_names_out()
display(X.shape)
terms

In [ ]:
article_row = X[1]

IDF_article = np.log(X.shape[0] / np.asarray((X > 0).sum(axis=0))).ravel()

In [ ]:
cv = CountVectorizer(vocabulary=terms)
counts = cv.transform(processed_docs).toarray()

TF_article = counts[1] / counts[1].sum()

In [ ]:
TF_IDF_article = TF_article * IDF_article
TF_IDF_article

In [ ]:
for t in  np.argsort(TF_IDF_article)[::-1][:15]:
    print(f"'{terms[t]}'      with TF-IDF = {TF_IDF_article[t]:.3f}")

### b.      Create a word cloud of the 20 most frequent words in this article (the 2nd article in the crude corpus).

In [ ]:
all_ = re.findall(r"[A-Za-z]+", article)
freq_all = Counter(all_)

top_all = dict(freq_all.most_common(20))
top_all

In [ ]:
cloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate_from_frequencies(top_all)

plt.figure(figsize=(10, 5))
plt.imshow(cloud)
plt.axis("off")

plt.savefig('resources/cloud_freq_raw.svg', format='svg', bbox_inches='tight')

plt.show()

### c.       Again, create a word cloud of the 20 most frequent words in the article. But this time, remove all stop words, punctuations, and numbers.

In [ ]:
top =  np.argsort(counts[1])[::-1][:20]
freq = {terms[i]: int(counts[1][i]) for i in top}
freq

In [ ]:
cloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate_from_frequencies(freq)

plt.figure(figsize=(10, 5))
plt.imshow(cloud)
plt.axis("off")

plt.savefig('resources/cloud_freq_proc.svg', format='svg', bbox_inches='tight')

plt.show()

### d.      Create a word cloud of 20 words in this article with the highest TF-IDF values.

In [ ]:
docs = [reuters.raw(ids[0]), article] + [reuters.raw(i) for i in ids[1:]]

vec_raw = TfidfVectorizer(token_pattern=r"\b[a-zA-Z]{3,}\b")
X_raw = vec_raw.fit_transform(docs)
terms_raw = vec_raw.get_feature_names_out()

articl_row = X_raw[1].toarray().ravel()
articl_top = np.argsort(articl_row)[::-1][:20]
freq_raw = {terms_raw[i]: float(articl_row[i]) for i in articl_top}
freq_raw

In [ ]:
cloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate_from_frequencies(freq_raw)

plt.figure(figsize=(10, 5))
plt.imshow(cloud)
plt.axis("off")

plt.savefig('resources/cloud_TF-IDF_raw.svg', format='svg', bbox_inches='tight')

plt.show()

### e.      Create a word cloud of 20 words in this article with the highest TF-IDF values, removing all stop words, punctuations, and numbers.

In [ ]:
top = np.argsort(TF_IDF_article)[::-1][:20]
freq = {terms[i]: float(TF_IDF_article[i]) for i in top}
freq

In [ ]:
cloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate_from_frequencies(freq)

plt.figure(figsize=(10, 5))
plt.imshow(cloud)
plt.axis("off")

plt.savefig('resources/cloud_TF-IDF_proc.jpg', format='svg', bbox_inches='tight')

plt.show()

### f.        Compare the 4 word clouds and the word list you came up in part (a). What have you found? Report your observations, and submit the report along with the word clouds. 


Сравнение:
| | Raw | Processed|
| :-: | :-: | :-: |
| frequency | ![](resources/cloud_freq_raw.svg) | ![](resources/cloud_freq_proc.svg) |
| TF-IDF | ![](resources/cloud_TF-IDF_raw.svg) | ![](resources/cloud_TF-IDF_proc.svg) |
----